# 🏦 Loan Approval Prediction
## End-to-End Machine Learning Project

**Model:** Logistic Regression  
**Problem Type:** Binary Classification  
**Prediction:** Loan Approved / Loan Rejected

This notebook follows the required workflow:

**Kaggle Dataset → EDA → Data Cleaning → Feature Engineering → Feature Selection → Train/Test Split → Model Building → Evaluation → Prediction**


## 1. Import Libraries


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report,
    roc_auc_score, roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from features import clean_strings, engineer_features, RAW_FEATURES

pd.set_option("display.max_columns", None)


## 2. Dataset Collection

Dataset: **Loan Approval Prediction Dataset** from Kaggle.

Expected file:

`data/loan_approval_dataset.csv`


In [ ]:
DATA_PATH = Path("data/loan_approval_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset missing. Put loan_approval_dataset.csv inside the data folder."
    )

df = pd.read_csv(DATA_PATH)
df = clean_strings(df)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 3. Data Understanding


In [ ]:
display(df.head())
display(df.tail())

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types / non-null counts:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 4. Minimum EDA Checklist


In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("\nNumerical columns:", numerical_columns)
print("\nCategorical columns:", categorical_columns)


### Target Variable Analysis


In [ ]:
df["loan_status"] = df["loan_status"].astype(str).str.strip()

display(df["loan_status"].value_counts())
display((df["loan_status"].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="loan_status")
plt.title("Loan Status Distribution")
plt.xlabel("Loan Status")
plt.ylabel("Count")
plt.show()


### Univariate Analysis — Numerical Distributions


In [ ]:
plot_cols = [
    "income_annum", "loan_amount", "loan_term", "cibil_score",
    "residential_assets_value", "commercial_assets_value",
    "luxury_assets_value", "bank_asset_value"
]

for col in plot_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()


### Outlier Analysis


In [ ]:
for col in plot_cols:
    plt.figure(figsize=(7, 2.8))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot: {col}")
    plt.show()


### Bivariate Analysis


In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="loan_status", y="cibil_score")
plt.title("CIBIL Score vs Loan Status")
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="loan_status", y="loan_amount")
plt.title("Loan Amount vs Loan Status")
plt.show()

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="education", hue="loan_status")
plt.title("Education vs Loan Status")
plt.show()

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="self_employed", hue="loan_status")
plt.title("Self Employment vs Loan Status")
plt.show()


### Correlation Analysis


In [ ]:
corr_df = df.select_dtypes(include=np.number).drop(columns=["loan_id"], errors="ignore")
plt.figure(figsize=(12, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


## 5. Data Cleaning

Cleaning steps:

1. Strip whitespace from column names and text values.
2. Remove duplicate rows.
3. Treat negative asset values as invalid and clip them to zero.
4. Missing numerical values will be imputed with the median.
5. Missing categorical values will be imputed with the most frequent value.


In [ ]:
df = df.drop_duplicates().copy()

asset_columns = [
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]

print("Negative asset values before cleaning:")
for col in asset_columns:
    print(col, int((df[col] < 0).sum()))

for col in asset_columns:
    df[col] = df[col].clip(lower=0)

print("\nRows after duplicate removal:", len(df))


## 6. Feature Engineering and Feature Selection

### Engineered features
- **total_assets** = sum of all asset values
- **loan_to_income_ratio** = loan amount / annual income

### Feature selection
`loan_id` is removed because it is only a unique identifier and does not describe applicant financial behaviour.


In [ ]:
target_map = {"Approved": 1, "Rejected": 0}
y = df["loan_status"].map(target_map)

if y.isna().any():
    raise ValueError("Unexpected value found in loan_status.")

X = df.drop(columns=["loan_status", "loan_id"])
X = engineer_features(X)

print("Selected / engineered features:")
print(X.columns.tolist())

display(X.head())


## 7. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 8. Preprocessing


In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 9. Logistic Regression Model


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

model.fit(X_train, y_train)
print("Logistic Regression training completed successfully.")


## 10. Model Evaluation


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Rejected", "Approved"],
    zero_division=0
))


### Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Rejected", "Approved"],
    yticklabels=["Rejected", "Approved"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


### ROC Curve and ROC-AUC


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


## 11. Prediction Probability and Classification Threshold

Logistic Regression predicts probabilities.

This project uses the standard **0.50 threshold**:

- Approval probability ≥ 0.50 → **Approved**
- Approval probability < 0.50 → **Rejected**

Changing the threshold changes the balance between precision and recall.


In [ ]:
probability_preview = pd.DataFrame({
    "Actual": y_test.head(10).map({0: "Rejected", 1: "Approved"}).values,
    "Approval_Probability": y_prob[:10],
    "Predicted": pd.Series(y_pred[:10]).map({0: "Rejected", 1: "Approved"}).values
})
display(probability_preview)


## 12. Prediction on Unseen / New Applicant Data


In [ ]:
new_applicant = pd.DataFrame([{
    "no_of_dependents": 2,
    "education": "Graduate",
    "self_employed": "No",
    "income_annum": 5000000,
    "loan_amount": 12000000,
    "loan_term": 10,
    "cibil_score": 750,
    "residential_assets_value": 6000000,
    "commercial_assets_value": 3000000,
    "luxury_assets_value": 7000000,
    "bank_asset_value": 4000000,
}])

new_applicant = engineer_features(new_applicant)

new_prediction = int(model.predict(new_applicant)[0])
new_probability = float(model.predict_proba(new_applicant)[0, 1])

print("Prediction:", "Approved" if new_prediction == 1 else "Rejected")
print(f"Approval probability: {new_probability:.2%}")


## 13. Save Trained Model and Metadata


In [ ]:
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "loan_pipeline.joblib"
METADATA_PATH = MODEL_DIR / "metadata.json"

joblib.dump(model, MODEL_PATH)

metadata = {
    "target_mapping": {"Rejected": 0, "Approved": 1},
    "threshold": 0.5,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "confusion_matrix": cm.tolist()
    }
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("SUCCESS — model files created:")
print(MODEL_PATH)
print(METADATA_PATH)

assert MODEL_PATH.exists(), "Model file was not created."
assert METADATA_PATH.exists(), "Metadata file was not created."
print("\nVerified: both files exist.")


## 14. Conclusion

After running the notebook, write a short conclusion using your actual results:

- Dataset contained 4,269 loan applications before any cleaning changes.
- The target was loan approval/rejection.
- `loan_id` was removed because it is an identifier.
- Negative asset anomalies were treated.
- Categorical features were encoded.
- Numerical features were scaled.
- `total_assets` and `loan_to_income_ratio` were engineered.
- Logistic Regression was evaluated using Accuracy, Precision, Recall, F1-score, Confusion Matrix and ROC-AUC.
- The trained model can predict unseen applicant data and output an approval probability.
